In [ ]:
# =============================================================================
# jobs/24_backfill_mp_briefing_drafts.py
# Draft MP briefings for EARLIER Flood Guidance Statements, so they can be
# reviewed after the event. DRAFTS ONLY. Nothing in this notebook sends anything.
#
# Notebook 20 drafts for the latest statement only, from constituency_flood_
# situation, which holds one statement. This notebook rebuilds the same picture
# for older statements from fgs_constituency_intersections, which the pipeline
# keeps for every statement, and writes drafts in the same wording to the same
# table (mp_briefing_drafts), with status = 'draft'.
#
# WHAT IS DIFFERENT FROM A DRAFT MADE AT THE TIME
#   - Warnings and alerts IN FORCE are not kept historically, so a backfilled
#     draft leaves that sentence out. It does say how many flood warning and
#     flood alert areas sat under the forecast (a potential measure, from
#     fgs_ea_area_intersections, which is kept per statement).
#   - The MP is today's MP for that constituency, matched on name.
#   - "Higher than in the previous statement" comes from fgs_constituency_changes
#     (notebook 18). A statement 18 never compared leaves that sentence out.
#   - created_at is the time this notebook ran, not the time of the statement.
#     Use issued_at in the body, or statement_id, to see which statement it is.
#
# SAFE TO RE-RUN. A constituency that already has a row for a statement, in any
# status, is skipped, so a draft a person has edited or moved on is never touched.
# The latest complete statement is skipped too: that is notebook 20's job, and it
# also knows the warnings in force.
#
# Parameters (widgets):
#   statements   "30" (default): the last 30 complete statements before the latest
#                "30d":          every complete statement issued in the last 30 days
#                "all":          every complete statement before the latest
#                "4811,4812":    exactly these statement ids (a single id works too)
#
# Exit values: "success" (drafts written, possibly none).
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from datetime import timedelta

import pandas as pd
from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT, TBL_CONSTITUENCIES,
    TBL_MP_CONTACTS, TBL_FGS_EA_INTERSECT, TBL_EA_CONST_LOOKUP, TBL_FGS_CHANGES,
    TBL_FGS_CHANGES_LOG, TBL_MP_BRIEFING_DRAFTS, BRIEFING_MIN_RISK_RANK, FGS_GEOMETRY_SOURCE,
    FGS_CONTACT_MIN_PCT, WARNING_CONTACT_MIN_PCT
)
from utils.helpers import (
    get_spark, utc_now, table_exists, risk_rank_sql,
    source_predicate, areas_under_forecast_sentence, BRIEFING_DRAFT_SCHEMA
)
from utils.briefing import (
    RANK_NAME, RANK_OF, build_draft, day_dates, optional_bool, situation_from_intersections,
    check_setting_is_current
)

# A session that was open before config.py changed still holds the old value. Stop rather
# than draft against it.
check_setting_is_current(os.path.join(PROJECT_ROOT, "config.py"), "BRIEFING_MIN_RISK_RANK", BRIEFING_MIN_RISK_RANK)

dbutils.widgets.text("statements", "30", "Statements: a number, all, or ids")
requested = dbutils.widgets.get("statements").strip().lower() or "30"


# =============================================================================
# SETUP: WHICH STATEMENTS
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT,
              TBL_CONSTITUENCIES, TBL_MP_CONTACTS):
    if not table_exists(spark, table):
        raise Exception(f"{table} does not exist. Run 01/02, 04, 05 and 07 first.")

log_in_source = source_predicate(spark, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, alias="l")
complete = spark.sql(f"""
    SELECT l.statement_id, s.issued_at
    FROM {TBL_FGS_INTERSECT_LOG} l
    JOIN {TBL_STATEMENTS} s
      ON s.statement_id = l.statement_id
     AND COALESCE(s.last_modified_at, '') = COALESCE(l.last_modified_at, '')
    WHERE {log_in_source}
    ORDER BY s.issued_at DESC, l.statement_id DESC
""").collect()
if not complete:
    raise Exception(
        f"No statement has up-to-date intersections for geometry source "
        f"'{FGS_GEOMETRY_SOURCE}'. Run 07_compute_intersections first."
    )

latest_id = int(complete[0]["statement_id"])
older = [r for r in complete if int(r["statement_id"]) != latest_id]   # newest first

window_start = None
if requested == "all":
    targets = older
elif requested.endswith("d") and requested[:-1].isdigit():
    window_start = (utc_now() - timedelta(days=int(requested[:-1]))).isoformat()[:19]
    targets = [r for r in older if str(r["issued_at"])[:19] >= window_start]       # the last N days
elif "," in requested or (requested.isdigit() and int(requested) > 1000):
    wanted = {int(x) for x in requested.replace(" ", "").split(",") if x}      # exact statement ids
    targets = [r for r in older if int(r["statement_id"]) in wanted]
    missing = wanted - {int(r["statement_id"]) for r in targets}
    if missing:
        print(f"Skipped (not complete, or the latest, which notebook 20 handles): {sorted(missing)}")
else:
    targets = older[:int(requested)]                                           # the last N before the latest

# Statements in the period that cannot be drafted for because their intersections are
# missing or out of date. Without this a quiet result looks like "nothing at risk".
if window_start is not None or requested.isdigit():
    cutoff = window_start or (str(targets[-1]["issued_at"])[:19] if targets else "")
    complete_ids = {int(r["statement_id"]) for r in complete}
    not_ready = [(int(r["statement_id"]), str(r["issued_at"])[:10]) for r in spark.sql(
        f"SELECT statement_id, issued_at FROM {TBL_STATEMENTS}").collect()
        if str(r["issued_at"])[:19] >= cutoff and int(r["statement_id"]) not in complete_ids]
    if not_ready:
        print(f"{len(not_ready)} statement(s) in this period have no up-to-date intersections, so "
              f"cannot be drafted for. Run 07_compute_intersections (recompute_all) first: "
              f"{sorted(not_ready)[:10]}")

targets = sorted(targets, key=lambda r: (str(r["issued_at"]), int(r["statement_id"])))
if not targets:
    print("Nothing to backfill.")
    dbutils.notebook.exit("success")

ids = [int(r["statement_id"]) for r in targets]
issued = {int(r["statement_id"]): str(r["issued_at"]) for r in targets}
id_list = ", ".join(str(i) for i in ids)
print(f"Latest complete statement {latest_id} is left to notebook 20.")
print(f"Backfilling {len(ids)} statement(s): {ids[0]} (issued {issued[ids[0]][:10]}) to "
      f"{ids[-1]} (issued {issued[ids[-1]][:10]}). Drafting for {RANK_NAME[BRIEFING_MIN_RISK_RANK]} or higher.")

In [ ]:
# =============================================================================
# LOAD WHAT THE DRAFTS NEED, FOR ALL THE STATEMENTS AT ONCE
# =============================================================================

i_in_source = source_predicate(spark, TBL_FGS_CONST_INTERSECT, FGS_GEOMETRY_SOURCE, alias="i")
inter_all = spark.sql(f"""
    SELECT i.statement_id, i.constituency_id, i.forecast_date, i.day_index, i.source,
           i.risk_level, i.intersection_pct, i.impact_label, i.likelihood_label
    FROM {TBL_FGS_CONST_INTERSECT} i
    WHERE i.statement_id IN ({id_list})
      AND {i_in_source}
""").toPandas()
print(f"Intersection rows: {len(inter_all):,}")

all_const = spark.sql(f"SELECT constituency_id, name AS constituency_name FROM {TBL_CONSTITUENCIES}").toPandas()
mp = spark.sql(f"SELECT constituency_name, member_id, name, email FROM {TBL_MP_CONTACTS}").toPandas()

# Which statements 18 compared, and which constituencies rose. A statement 18 never
# compared has escalation None (unknown), not False.
escalated = {}
if table_exists(spark, TBL_FGS_CHANGES_LOG) and table_exists(spark, TBL_FGS_CHANGES):
    compared = {int(r["statement_id"]) for r in spark.sql(
        f"SELECT DISTINCT statement_id FROM {TBL_FGS_CHANGES_LOG} WHERE statement_id IN ({id_list})").collect()}
    rose = spark.sql(f"""
        SELECT DISTINCT statement_id, constituency_id FROM {TBL_FGS_CHANGES}
        WHERE statement_id IN ({id_list}) AND change_type IN ('new', 'escalated')
    """).toPandas()
    for sid in ids:
        escalated[sid] = (set(rose.loc[rose["statement_id"] == sid, "constituency_id"])
                          if sid in compared else None)
else:
    print("fgs_changes_log or fgs_constituency_changes is missing (notebook 18), so no draft "
          "says whether the risk rose.")
    escalated = {sid: None for sid in ids}
print(f"Statements 18 compared: {sum(v is not None for v in escalated.values())} of {len(ids)}")

# Flood warning and alert areas under the forecast polygons, for the worst-risk drafts only.
areas_all = pd.DataFrame(columns=["statement_id", "constituency_id", "source", "risk_rank",
                                  "ea_area_code", "ea_area_type"])
if table_exists(spark, TBL_FGS_EA_INTERSECT) and table_exists(spark, TBL_EA_CONST_LOOKUP):
    e_in_source = source_predicate(spark, TBL_FGS_EA_INTERSECT, FGS_GEOMETRY_SOURCE, alias="e")
    areas_all = spark.sql(f"""
        SELECT DISTINCT e.statement_id, l.constituency_id, e.source,
                        {risk_rank_sql('e.risk_level')} AS risk_rank,
                        e.ea_area_code, LOWER(e.ea_area_type) AS ea_area_type
        FROM {TBL_FGS_EA_INTERSECT} e
        JOIN {TBL_EA_CONST_LOOKUP} l ON l.ea_area_code = e.ea_area_code
        WHERE e.statement_id IN ({id_list})
          AND {e_in_source}
          AND l.intersection_pct >= {WARNING_CONTACT_MIN_PCT}
          AND {risk_rank_sql('e.risk_level')} >= {BRIEFING_MIN_RISK_RANK}
    """).toPandas()
else:
    print("fgs_ea_area_intersections or ea_area_constituency_lookup is missing, so drafts "
          "say nothing about areas under the forecast.")

# Rows that already exist: never overwritten, in any status.
existing = set()
if table_exists(spark, TBL_MP_BRIEFING_DRAFTS):
    existing = {(int(r["statement_id"]), r["constituency_id"]) for r in spark.sql(f"""
        SELECT statement_id, constituency_id FROM {TBL_MP_BRIEFING_DRAFTS}
        WHERE statement_id IN ({id_list})
          AND COALESCE(geometry_source, 'risk_areas') = '{FGS_GEOMETRY_SOURCE}'
    """).collect()}
print(f"Drafts that already exist for these statements (kept as they are): {len(existing)}")

In [ ]:
# =============================================================================
# BUILD AND WRITE THE DRAFTS
# =============================================================================

rows = []
per_statement = []
for sid in ids:
    inter = inter_all[inter_all["statement_id"] == sid]
    situation = situation_from_intersections(
        inter, all_const, mp, escalated[sid], sid, issued[sid], FGS_CONTACT_MIN_PCT)
    at_risk = situation[situation["fgs_worst_rank"] >= BRIEFING_MIN_RISK_RANK]
    candidates = at_risk[at_risk["mp_matched"]]

    # The same overlap floor job 14 applies, so the impact and likelihood lines match a live draft.
    inter = inter[inter["intersection_pct"].isna() | (inter["intersection_pct"] >= FGS_CONTACT_MIN_PCT)]
    inter = inter.assign(risk_rank=inter["risk_level"].map(RANK_OF).fillna(0).astype(int))
    days = day_dates(zip(inter["day_index"], inter["forecast_date"]))
    detail = {k: g.to_dict("records") for k, g in inter.groupby("constituency_id")}
    areas = areas_all[areas_all["statement_id"] == sid]
    areas_by = {k: g.to_dict("records") for k, g in areas.groupby("constituency_id")}

    made = 0
    for _, c in candidates.iterrows():
        cid = c["constituency_id"]
        if (sid, cid) in existing:
            continue
        subject, body = build_draft(c, detail.get(cid, []), areas_by.get(cid, []), days,
                                    areas_under_forecast_sentence)
        rows.append((
            sid, FGS_GEOMETRY_SOURCE, cid, c["constituency_name"],
            int(c["mp_member_id"]), c["mp_name"], c["mp_email"],
            int(c["fgs_worst_rank"]), RANK_NAME[int(c["fgs_worst_rank"])], str(c["fgs_worst_date"])[:10],
            optional_bool(c["fgs_escalated"]), None,
            subject, body, "draft", None, None, now_iso,
        ))
        made += 1
    per_statement.append((sid, issued[sid][:10], len(at_risk), len(at_risk) - len(candidates), made))

print(f"\n{'statement':>10}  {'issued':10}  {'at risk':>7}  {'no MP':>5}  {'drafted':>7}")
for sid, when, n_risk, n_no_mp, n_made in per_statement:
    print(f"{sid:>10}  {when:10}  {n_risk:>7}  {n_no_mp:>5}  {n_made:>7}")

if rows:
    (
        spark.createDataFrame(rows, BRIEFING_DRAFT_SCHEMA).write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(TBL_MP_BRIEFING_DRAFTS)
    )
elif not table_exists(spark, TBL_MP_BRIEFING_DRAFTS):
    # Nothing to draft, but make the table exist so the app does not report it missing.
    (
        spark.createDataFrame([], BRIEFING_DRAFT_SCHEMA).write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_MP_BRIEFING_DRAFTS)
    )

print(f"\nDrafts written: {len(rows)}")
dbutils.notebook.exit("success")